
# Silver — URA

Atualiza a Silver de URA consumindo a Bronze (append-only, **já estruturada**) como
**fonte de streaming Delta** (`Trigger.AvailableNow`) e aplicando **MERGE** idempotente
via `foreachBatch`. O **checkpoint** controla o progresso; como `foreachBatch` tem
semântica **at-least-once**, a não-duplicidade vem do **MERGE por chave de negócio**.

**Fluxo**
1. Lê a Bronze como stream Delta, a partir do checkpoint.
2. Valida o **contrato** de campos obrigatórios; inválidos vão à quarentena (idempotente).
3. Projeta/renomeia os campos estruturados e deriva datas (helpers da lib) — sem `from_json`.
4. Aplica um **gate de Data Quality** (registra em `__dq_results`) antes do MERGE.
5. Garante a Silver com o mesmo schema e aplica **MERGE** por chave de negócio.



## Parâmetros


In [ ]:
import sys

# Em execução interativa, torna a lib importável a partir do Repos; nos jobs a lib
# é instalada no cluster via wheel (ver `libraries` no databricks.yml).
sys.path.append("/Workspace/Repos/data_master/Databricks/lib")

dbutils.widgets.text("catalog", "prd")
dbutils.widgets.text("bronze_schema", "b_dm_callcenter")
dbutils.widgets.text("silver_schema", "s_dm_callcenter")
dbutils.widgets.text("bronze_table", "ura_once")
dbutils.widgets.text("silver_table", "tabe_ura_anlt")
dbutils.widgets.text("checkpoint_base", "/Volumes/prd/s_dm_callcenter/checkpoints/silver")

CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
BRONZE_TABLE  = dbutils.widgets.get("bronze_table")
SILVER_TABLE  = dbutils.widgets.get("silver_table")

BRONZE_FQN = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_FQN = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
CHECKPOINT = f"{dbutils.widgets.get('checkpoint_base').rstrip('/')}/{SILVER_TABLE}"
DQ_RESULTS = f"{CATALOG}.{SILVER_SCHEMA}.__dq_results"
QUARANTINE = f"{CATALOG}.{SILVER_SCHEMA}.__quarantine"
METRICS    = f"{CATALOG}.{SILVER_SCHEMA}.__dataset_metrics"

from pyspark.sql import functions as F
from transforms import SilverStream, rename_columns, add_period_and_dates, add_load_audit
from transforms.contracts import contract_for
from quality import Expectation, run_observability

CONTRACT = contract_for(BRONZE_TABLE)

print("Bronze:", BRONZE_FQN)
print("Silver:", SILVER_FQN)
print("Checkpoint:", CHECKPOINT)


## Transformação


In [ ]:
def transform(df):
    rename = {
        "id_chamada":          "ID_CHAM",
        "id_cliente":          "ID_CLIE",
        "id_fila":             "ID_FILA",
        "data_hora_inicio":    "DH_INIC",
        "data_hora_fim":       "DH_FIM",
        "opcoes_navegadas":    "QT_OPCA_NAVG",
        "codigo_opcao":        "CD_ULTI_OPCA",
        "autenticado":         "IN_AUTN",
        "derivado_atendimento":"IN_DERV_ATEN",
    }
    df = rename_columns(df, rename)
    df = add_period_and_dates(df, "DH_INIC")
    df = df.withColumn("DT_FIM", F.to_date("DH_FIM"))
    df = add_load_audit(df)
    return df


## ▶️ Execução
Lê a Bronze como stream Delta (`AvailableNow`), transforma e aplica `MERGE` idempotente
via `foreachBatch`. O **checkpoint** controla o progresso.


In [ ]:
# Contrato (obrigatórios) via quarentena idempotente + gate de DQ por micro-batch. O
# checkpoint controla o progresso; a idempotência (at-least-once sem duplicar) vem do
# MERGE por chave.
checks = [
    Expectation.not_null("ID_CHAM"),
    Expectation.unique("ID_CHAM"),
    Expectation.not_null("ID_CLIE"),
    Expectation.not_null("DH_INIC"),
    Expectation.accepted_values("IN_AUTN", [True, False]),
]

stream = SilverStream(spark)
stream.run(
    source_table_fqn=BRONZE_FQN,
    target_table_fqn=SILVER_FQN,
    transform=transform,
    keys=["ID_CHAM"],
    checkpoint_location=CHECKPOINT,
    cluster_by=["CD_PERI", "DT_INIC", "ID_CHAM"],
    expectations=checks,
    dq_results_table=DQ_RESULTS,
    contract_required=CONTRACT.required,
    quarantine_table=QUARANTINE,
    schema_version=CONTRACT.version,
)
print(f"[OK] Silver atualizada → {SILVER_FQN}")

# Observabilidade de dataset: volume da data mais recente vs média histórica + freshness.
obs = run_observability(
    spark,
    target_table_fqn=SILVER_FQN,
    metrics_table=METRICS,
    date_col="DT_INIC",
    timestamp_col="DH_INIC",
    max_freshness_minutes=48 * 60,
)
print(obs.summary())
obs.raise_if_critical_failed()   # severidade padrão warn: sinaliza sem interromper